In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
%pip install -U dagshub mlflow skops --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 66.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import dagshub
dagshub.init(repo_owner='gvakh23', repo_name='ML_assignment2', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=44abdb49-271d-43a0-b4fc-faa974e3ed4b&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=93ffde4145611095387a1966197e62269423b5f40437d15a1a33bd79b59fec4f




Output()

Accessing as gvakh23

Initialized MLflow to track repo "gvakh23/ML_assignment2"

Repository gvakh23/ML_assignment2 initialized!

In [5]:
import mlflow
print(mlflow.get_tracking_uri())

https://dagshub.com/gvakh23/ML_assignment2.mlflow


In [6]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

train_trans  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_ident  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

train = train_trans.merge(train_ident, on='TransactionID', how='left')


In [7]:
train_sorted = train.sort_values('TransactionDT')

val_size = int(len(train_sorted) * 0.2)
train_df = train_sorted.iloc[:-val_size].copy()
val_df   = train_sorted.iloc[-val_size:].copy()

print(f"Train split : {train_df.shape}, fraud rate: {train_df['isFraud'].mean():.2%}")
print(f"Val   split : {val_df.shape},   fraud rate: {val_df['isFraud'].mean():.2%}")


Train split : (472432, 434), fraud rate: 3.51%
Val   split : (118108, 434),   fraud rate: 3.44%


In [8]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
class Cleaner(BaseEstimator, TransformerMixin):
    """
    Drops high-null columns, encodes M columns (T/F→1/0),
    fills remaining NaN with -999.
    All thresholds computed on train only.
    """
 
    def __init__(self, null_threshold=0.9, fill_value=-999):
        self.null_threshold = null_threshold
        self.fill_value     = fill_value
 
    def fit(self, X, y=None):
        print("clean_in")

        df = X.copy()
 
        # Learn which cols to drop (>null_threshold missing)
        null_rates = df.isnull().mean()
        self.cols_to_drop_ = null_rates[null_rates > self.null_threshold].index.tolist()
        self.cols_to_drop_ = [c for c in self.cols_to_drop_
                               if c not in ['isFraud', 'TransactionID']]
 
        # Learn M column names
        self.m_cols_ = [c for c in df.columns if c.startswith('M')]
 
        with mlflow.start_run(run_name='XGBoost_Cleaning', nested=True):
            mlflow.log_param('null_threshold',  self.null_threshold)
            mlflow.log_param('fill_value',      self.fill_value)
            mlflow.log_param('fill_reason',     'tree_model_out_of_range_sentinel')
            mlflow.log_param('m_col_encoding',  'T=1_F=0_NaN=fill_value')
            mlflow.log_metric('cols_dropped',   len(self.cols_to_drop_))
            mlflow.log_metric('cols_remaining', df.shape[1] - len(self.cols_to_drop_))
            mlflow.log_metric('rows',           df.shape[0])
            mlflow.log_metric('fraud_rate',     float(y.mean()) if y is not None else -1)
        print(f"[Cleaner] Done — dropped {len(self.cols_to_drop_)} cols, {df.shape[0]} rows")
        return self
 
    def transform(self, X):
        print("clean trans in")
        df = X.copy()
 
        # Drop high-null cols
        df.drop(columns=[c for c in self.cols_to_drop_ if c in df.columns],
                inplace=True, errors='ignore')
 
        # M columns T/F → 1/0
        for col in self.m_cols_:
            if col in df.columns:
                df[col] = df[col].map({'T': 1, 'F': 0})
 
        # Fill NaN
        df.fillna(self.fill_value, inplace=True)
 
        return df

In [10]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.d_cols      = ['D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D15']
        self.freq_cols   = ['card1', 'card2', 'addr1', 'uid', 'uid2', 'P_emaildomain']
        self.agg_cols    = ['card1', 'uid']          # removed card2/addr1 — less useful
        self.agg_targets = ['TransactionAmt', 'D1', 'C1', 'C2', 'C13']

    def fit(self, X, y=None):
        df = X.copy()

        # 1. D normalization mins
        self.d_min_map_ = {}
        for d in self.d_cols:
            if d in df.columns:
                self.d_min_map_[d] = df.groupby('card1')[d].min()

        # 2. Build UIDs
        df['uid']  = df['card1'].astype(str) + '_' + df['card2'].astype(str)
        # uid2: use D1 only if it's not -999 (missing)
        d1_str     = df['D1'].apply(lambda x: str(int(x)) if x != -999 else 'NA')
        df['uid2'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str) + '_' + d1_str

        # 3. Frequency maps
        self.freq_maps_ = {}
        for col in self.freq_cols:
            if col in df.columns:
                self.freq_maps_[col] = df[col].value_counts()

        # 4. Aggregation maps — store as dict of Series for fast map()
        self.agg_maps_ = {}
        for col in self.agg_cols:
            if col not in df.columns:
                continue
            self.agg_maps_[col] = {}
            for target in self.agg_targets:
                if target not in df.columns:
                    continue
                self.agg_maps_[col][target] = {}
                grp = df.groupby(col)[target]
                self.agg_maps_[col][target]['mean'] = grp.mean()
                self.agg_maps_[col][target]['std']  = grp.std()
                self.agg_maps_[col][target]['max']  = grp.max()

        # 5. UID label encoders — store as plain dict for fastest mapping
        self.uid_maps_ = {}
        for col in ['uid', 'uid2']:
            if col in df.columns:
                unique_vals = df[col].astype(str).unique()
                self.uid_maps_[col] = {v: i for i, v in enumerate(unique_vals)}

        # 6. Email suffix encoder
        if 'P_emaildomain' in df.columns:
            suffixes = df['P_emaildomain'].astype(str).str.split('.').str[-1]
            unique_sfx = suffixes.unique()
            self.email_suffix_map_ = {v: i for i, v in enumerate(unique_sfx)}
        else:
            self.email_suffix_map_ = {}

        return self

    def transform(self, X):
        df = X.copy()

        # ── Time features ─────────────────────────────────────
        df['hour']  = (df['TransactionDT'] / 3600) % 24
        df['day']   = (df['TransactionDT'] / (3600 * 24)) % 7

        # ── Log transform ─────────────────────────────────────
        df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)

        # ── Email features ────────────────────────────────────
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['same_email'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
            suffixes = df['P_emaildomain'].astype(str).str.split('.').str[-1]
            df['P_email_suffix'] = suffixes.map(self.email_suffix_map_).fillna(-1).astype(int)

        # ── D normalization ───────────────────────────────────
        for d, min_map in self.d_min_map_.items():
            if d in df.columns:
                df[f'{d}_norm'] = df[d] - df['card1'].map(min_map).fillna(0)

        # ── UIDs ──────────────────────────────────────────────
        df['uid']  = df['card1'].astype(str) + '_' + df['card2'].astype(str)
        d1_str     = df['D1'].apply(lambda x: str(int(x)) if x != -999 else 'NA')
        df['uid2'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str) + '_' + d1_str

        # ── Frequency encoding ────────────────────────────────
        for col, freq_map in self.freq_maps_.items():
            if col in df.columns:
                df[f'{col}_freq'] = df[col].map(freq_map).fillna(0)

        # ── Aggregations (fast: map Series directly) ──────────
        for col, targets in self.agg_maps_.items():
            if col not in df.columns:
                continue
            for target, stats in targets.items():
                for stat, series in stats.items():
                    df[f'{col}_{target}_{stat}'] = df[col].map(series).fillna(-999)

        # ── UID encoding (fast: plain dict map) ───────────────
        for col, uid_map in self.uid_maps_.items():
            if col in df.columns:
                df[col] = df[col].astype(str).map(uid_map).fillna(-1).astype(int)

        return df




In [12]:
import pandas as pd
import mlflow
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder

class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, ohe_threshold=10):
        self.ohe_threshold = ohe_threshold
        self.device_map_ = {'desktop': 1, 'mobile': 0}
        self.ohe_info_ = {}
        self.le_encoders_ = {}
        self.le_cols_ = []

    def fit(self, X, y=None):
        print("encode_in")

        df = X.copy()
        
        # 1. Identify Categorical Columns (excluding IDs/Targets)
        all_cat_cols = [c for c in df.select_dtypes(include='object').columns 
                        if c not in ['TransactionID', 'isFraud', 'DeviceType']]

        # 2. Split logic: OHE vs Label Encode based on threshold
        for col in all_cat_cols:
            n_unique = df[col].nunique()
            if n_unique < self.ohe_threshold:
                # Store unique categories for OHE to ensure same columns in transform
                self.ohe_info_[col] = df[col].unique().tolist()
            else:
                # Fit LabelEncoder for high-cardinality
                le = LabelEncoder()
                le.fit(df[col].astype(str))
                self.le_encoders_[col] = le
                self.le_cols_.append(col)

        with mlflow.start_run(run_name='XGBoost_Categorical_Encoding', nested=True):
            mlflow.log_param('ohe_threshold', self.ohe_threshold)
            mlflow.log_param('ohe_cols', list(self.ohe_info_.keys()))
            mlflow.log_param('le_cols', self.le_cols_)
            mlflow.log_metric('ohe_count', len(self.ohe_info_))
            mlflow.log_metric('le_count', len(self.le_cols_))
        print(f"[CategoricalEncoder] Done — OHE cols: {len(self.ohe_info_)}, LE cols: {len(self.le_cols_)}")

        return self

    def transform(self, X):
        print("encode trans in")

        df = X.copy()

        # ── BINARY: DeviceType ──
        if 'DeviceType' in df.columns:
            df['DeviceType'] = df['DeviceType'].map(self.device_map_).fillna(-999)

        # ── ONE-HOT ENCODING ──
        for col, categories in self.ohe_info_.items():
            if col in df.columns:
        
                dummies = pd.get_dummies(df[col], prefix=col)
        
                expected_cols = [f"{col}_{cat}" for cat in categories]
                dummies = dummies.reindex(columns=expected_cols, fill_value=0)
        
                df = pd.concat([df, dummies], axis=1)
                df.drop(columns=[col], inplace=True)
                
        # ── LABEL ENCODING ──
        for col, le in self.le_encoders_.items():
            if col in df.columns:
                # Handle unseen categories by mapping to -1
                known_classes = set(le.classes_)
                mapping = {cls: idx for idx, cls in enumerate(le.classes_)}
                df[col] = df[col].astype(str).map(mapping).fillna(-1)

        return df

In [13]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, variance_threshold=0.01):
        self.variance_threshold = variance_threshold
        self.drop_cols_ = []
        self.feature_cols_ = []

    def fit(self, X, y=None):
        print("select_in")

        df = X.copy()
        numeric_df = df.select_dtypes(include=[np.number])
        
        # ── METHOD 1: Variance Threshold ──
        vt = VarianceThreshold(threshold=self.variance_threshold)
        vt.fit(numeric_df)
        low_var_cols = numeric_df.columns[~vt.get_support()].tolist()

        # ── METHOD 2: V-Column NaN-Group Selection ──
        # In this dataset, V columns are often redundant. 
        # Group them by their 'NaN pattern' and keep only the one strongest with target.
        v_cols = [c for c in df.columns if c.startswith('V')]
        
        # Hash null pattern by count only — avoids 470k-element tuple keys
        null_patterns = {}
        for col in v_cols:
            pattern = int((df[col] == -999).sum())
            null_patterns.setdefault(pattern, []).append(col)

        best_per_group = []
        if y is not None:
            y_s = pd.Series(y.values if hasattr(y, 'values') else np.array(y), index=df.index)
            for pattern, cols in null_patterns.items():
                corrs = df[cols].corrwith(y_s).abs().fillna(0)
                best_per_group.append(corrs.idxmax())
        else:
            for pattern, cols in null_patterns.items():
                best_per_group.append(cols[0])

        v_to_drop = [c for c in v_cols if c not in best_per_group]
        
        # Combine all columns to remove
        self.drop_cols_ = list(set(low_var_cols + v_to_drop))
        
        # Save the "survivor" column names in order
        self.feature_cols_ = [c for c in df.columns if c not in self.drop_cols_]

        with mlflow.start_run(run_name='XGBoost_Feature_Selection', nested=True):
            mlflow.log_param('variance_threshold', self.variance_threshold)
            mlflow.log_metric('v_nan_groups', len(null_patterns))
            mlflow.log_metric('total_features_dropped', len(self.drop_cols_))
            mlflow.log_metric('final_feature_count', len(self.feature_cols_))
        print(f"[FeatureSelector] Done — dropped {len(self.drop_cols_)} features, {len(self.feature_cols_)} remaining")

        return self

    def transform(self, X):
        print("select trans in")

        df = X.copy()
        
        # 1. Drop the identified useless columns
        df.drop(columns=[c for c in self.drop_cols_ if c in df.columns], 
                inplace=True, errors='ignore')
        
        # 2. EXACT ALIGNMENT
        # This ensures the features going into XGBoost are exactly what it expects
        # If a column is missing in Val/Test, it adds it as 0.
        df = df.reindex(columns=self.feature_cols_, fill_value=0)
        
        return df

In [14]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

NULL_THRESH = 0.9
OHE_THRESH  = 10
VAR_THRESH  = 0.01


import mlflow
import mlflow.sklearn

mlflow.set_experiment("XGBoost_Training")

preprocessing = Pipeline([
    ('cleaning',     Cleaner(null_threshold=NULL_THRESH)),
    ('feature_eng',  FeatureEngineer()),
    ('encoding',     CategoricalEncoder(ohe_threshold=OHE_THRESH)),
    ('selection',    FeatureSelector(variance_threshold=VAR_THRESH)),
])



In [15]:
X_train = train_df.drop(columns=['isFraud', 'TransactionID'])
y_train = train_df['isFraud']

X_val = val_df.drop(columns=['isFraud', 'TransactionID'])
y_val = val_df['isFraud']


In [16]:
X_train = preprocessing.fit_transform(X_train, y_train)
X_val   = preprocessing.transform(X_val)

print("Preprocessing Done")

clean_in
🏃 View run XGBoost_Cleaning at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/fbe8b22354c04803a400d9b3045ae120
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1
[Cleaner] Done — dropped 12 cols, 472432 rows
clean trans in
encode_in
🏃 View run XGBoost_Categorical_Encoding at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/7b62fc3a89d049428d8a2f4081248e42
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1
[CategoricalEncoder] Done — OHE cols: 13, LE cols: 6
encode trans in
select_in
🏃 View run XGBoost_Feature_Selection at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/1a6fc7b0733d4ca9b6d94e573eea331a
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1
[FeatureSelector] Done — dropped 326 features, 180 remaining
select trans in
clean trans in
encode trans in
select trans in
Preprocessing Done


In [17]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit


In [18]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(scale_pos_weight)

27.46147358274595


In [19]:
with mlflow.start_run(run_name="XGBoost_Underfitted"):

    params = {
        'max_depth': 2,
        'n_estimators': 50,
        'learning_rate': 0.3,
        'scale_pos_weight': scale_pos_weight,
        'eval_metric': 'auc',
        'random_state': 42,
        # 'n_jobs': -1
        'tree_method': 'hist',
        'device': 'cuda'
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])

    mlflow.log_params(params)
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc",   val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)

    print(f"UNDERFITTED — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"Gap: {train_auc - val_auc:.4f} (both low = underfitting)")

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [19:19:46] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


UNDERFITTED — train: 0.8912, val: 0.8665
Gap: 0.0247 (both low = underfitting)
🏃 View run XGBoost_Underfitted at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/8e1b91c124824d3e8650ae679e053d15
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1


In [20]:
with mlflow.start_run(run_name="XGBoost_Overfitted"):

    params = {
        'max_depth': 15,
        'n_estimators': 500,
        'learning_rate': 0.3,
        'min_child_weight': 1,
        'subsample': 1.0,
        'colsample_bytree': 1.0,
        'reg_alpha': 0.0,
        'reg_lambda': 0.0,
        'scale_pos_weight': scale_pos_weight,
        'eval_metric': 'auc',
        'random_state': 42,
        # 'n_jobs': -1
        'tree_method': 'hist',
        'device': 'cuda'
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])

    mlflow.log_params(params)
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc",   val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)

    print(f"OVERFITTED  — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"Gap: {train_auc - val_auc:.4f} (high gap = overfitting)")



OVERFITTED  — train: 1.0000, val: 0.9070
Gap: 0.0930 (high gap = overfitting)
🏃 View run XGBoost_Overfitted at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/3f3eb43000f54719929db17ea191548a
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1


In [21]:
with mlflow.start_run(run_name="XGBoost_Baseline") as run:

    params = {
        'max_depth': 6,
        'n_estimators': 500,
        'learning_rate': 0.05,
        'min_child_weight': 10,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'scale_pos_weight': scale_pos_weight,
        'eval_metric': 'auc',
        'random_state': 42,
        # 'n_jobs': -1
        'tree_method': 'hist',
        'device': 'cuda'
    }

    model = XGBClassifier(**params)

    # TimeSeriesSplit cross-validation on train split only
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []

    for fold, (tr_idx, cv_idx) in enumerate(tscv.split(X_train)):
        X_tr  = X_train.iloc[tr_idx]
        y_tr  = y_train.iloc[tr_idx]
        X_cv  = X_train.iloc[cv_idx]
        y_cv  = y_train.iloc[cv_idx]

        fold_model = XGBClassifier(**params)
        fold_model.fit(X_tr, y_tr)
        score = roc_auc_score(y_cv, fold_model.predict_proba(X_cv)[:, 1])
        cv_scores.append(score)
        print(f"  Fold {fold+1}: {score:.4f}")

    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])

    mlflow.log_params(params)
    mlflow.log_metric("train_auc",    train_auc)
    mlflow.log_metric("val_auc",      val_auc)
    mlflow.log_metric("cv_mean_auc",  np.mean(cv_scores))
    mlflow.log_metric("cv_std_auc",   np.std(cv_scores))
    mlflow.log_metric("overfit_gap",  train_auc - val_auc)

    baseline_run_id = run.info.run_id

    print(f"\nBASELINE    — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"CV scores   — mean: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}")
    print(f"Gap         — {train_auc - val_auc:.4f}")

  Fold 1: 0.8940
  Fold 2: 0.9092
  Fold 3: 0.9138
  Fold 4: 0.9088
  Fold 5: 0.9252

BASELINE    — train: 0.9824, val: 0.9167
CV scores   — mean: 0.9102, std: 0.0101
Gap         — 0.0658
🏃 View run XGBoost_Baseline at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/f132c7053398480f949e0c640bb6ab82
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1


In [22]:
with mlflow.start_run(run_name="XGBoost_Tuned") as run:
    # Changes vs baseline:
    # - lower learning_rate (0.02 vs 0.05) → slower but more stable
    # - higher min_child_weight (20 vs 10) → less overfitting
    # - lower subsample (0.7 vs 0.8) → more regularization
    # - higher reg_alpha (0.5 vs 0.1) → more L1 regularization
    params = {
        'max_depth': 6,
        'n_estimators': 1000,
        'learning_rate': 0.02,
        'min_child_weight': 20,
        'subsample': 0.7,
        'colsample_bytree': 0.7,
        'reg_alpha': 0.5,
        'reg_lambda': 2.0,
        'scale_pos_weight': scale_pos_weight,
        'eval_metric': 'auc',
        'random_state': 42,
        'tree_method': 'hist',
        'device': 'cuda'
    }

    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []

    for fold, (tr_idx, cv_idx) in enumerate(tscv.split(X_train)):
        fold_model = XGBClassifier(**params)
        fold_model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        score = roc_auc_score(
            y_train.iloc[cv_idx],
            fold_model.predict_proba(X_train.iloc[cv_idx])[:, 1]
        )
        cv_scores.append(score)
        print(f"  Fold {fold+1}: {score:.4f}")

    model_tuned = XGBClassifier(**params)
    model_tuned.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model_tuned.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model_tuned.predict_proba(X_val)[:, 1])

    mlflow.log_params(params)
    mlflow.log_metric("train_auc",   train_auc)
    mlflow.log_metric("val_auc",     val_auc)
    mlflow.log_metric("cv_mean_auc", np.mean(cv_scores))
    mlflow.log_metric("cv_std_auc",  np.std(cv_scores))
    mlflow.log_metric("overfit_gap", train_auc - val_auc)

    tuned_run_id = run.info.run_id

    print(f"\nTUNED    — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"CV       — mean: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}")
    print(f"Gap      — {train_auc - val_auc:.4f}")

  Fold 1: 0.9013
  Fold 2: 0.9148
  Fold 3: 0.9148
  Fold 4: 0.9110
  Fold 5: 0.9269

TUNED    — train: 0.9767, val: 0.9170
CV       — mean: 0.9137, std: 0.0082
Gap      — 0.0598
🏃 View run XGBoost_Tuned at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/cb6f2670e60a4cbda0d99de8a3d7f638
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1


In [23]:
with mlflow.start_run(run_name="XGBoost_NoClassWeight"):

    params = {
        'max_depth': 5, 'n_estimators': 1000, 'learning_rate': 0.02,
        'min_child_weight': 20, 'subsample': 0.7, 'colsample_bytree': 0.7,
        'reg_alpha': 0.5, 'reg_lambda': 2.0,
        'scale_pos_weight': 1,
        'eval_metric': 'auc', 'random_state': 42,
        'tree_method': 'hist', 'device': 'cuda'
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])

    mlflow.log_params(params)
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc",   val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)

    print(f"NO CLASS WEIGHT — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print("Compare val_auc vs Tuned run to see impact of scale_pos_weight=28")

NO CLASS WEIGHT — train: 0.9447, val: 0.9043
Compare val_auc vs Tuned run to see impact of scale_pos_weight=28
🏃 View run XGBoost_NoClassWeight at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/d81430a968ac4032bde805ac5ee97482
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1


In [24]:
with mlflow.start_run(run_name="XGBoost_EarlyStopping") as run:

    params = {
        'max_depth': 6,
        'n_estimators': 5000, 
        'learning_rate': 0.02,
        'min_child_weight': 16,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.3,
        'reg_lambda': 1.5,
        'scale_pos_weight': scale_pos_weight,
        'eval_metric': 'auc',
        'random_state': 42,
        'tree_method': 'hist',
        'device': 'cuda'
    }

    model = XGBClassifier(
        **params,
        early_stopping_rounds=100
    )

    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100
    )

    best_iteration = model.best_iteration
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])

    mlflow.log_params(params)
    mlflow.log_metric("train_auc",      train_auc)
    mlflow.log_metric("val_auc",        val_auc)
    mlflow.log_metric("overfit_gap",    train_auc - val_auc)
    mlflow.log_metric("best_iteration", best_iteration)

    print(f"\nEARLY STOPPING — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"Best iteration: {best_iteration} (out of 3000)")
    print(f"Gap: {train_auc - val_auc:.4f}")

[0]	validation_0-auc:0.80999
[100]	validation_0-auc:0.87489
[200]	validation_0-auc:0.89209
[300]	validation_0-auc:0.90071
[400]	validation_0-auc:0.90498
[500]	validation_0-auc:0.90874
[600]	validation_0-auc:0.91073
[700]	validation_0-auc:0.91257
[800]	validation_0-auc:0.91380
[900]	validation_0-auc:0.91485
[1000]	validation_0-auc:0.91653
[1100]	validation_0-auc:0.91707
[1200]	validation_0-auc:0.91774
[1300]	validation_0-auc:0.91807
[1400]	validation_0-auc:0.91859
[1500]	validation_0-auc:0.91869
[1600]	validation_0-auc:0.91936
[1700]	validation_0-auc:0.91911
[1701]	validation_0-auc:0.91914

EARLY STOPPING — train: 0.9875, val: 0.9194
Best iteration: 1601 (out of 3000)
Gap: 0.0682
🏃 View run XGBoost_EarlyStopping at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/b6496ee705b0443a9efad3a5862c3937
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1


In [25]:
with mlflow.start_run(run_name="XGBoost_Undersampling"):

    from imblearn.under_sampling import RandomUnderSampler

    print(f"Before — fraud: {y_train.sum():,}, non-fraud: {(y_train==0).sum():,}")

    rus = RandomUnderSampler(
        sampling_strategy=0.1,
        random_state=42
    )
    X_train_us, y_train_us = rus.fit_resample(X_train, y_train)

    print(f"After  — fraud: {y_train_us.sum():,}, non-fraud: {(y_train_us==0).sum():,}")
    print(f"New shape: {X_train_us.shape}")

    params = {
        'max_depth': 6, 'n_estimators': 3772,
        'learning_rate': 0.01, 'min_child_weight': 20,
        'subsample': 0.8, 'colsample_bytree': 0.8,
        'reg_alpha': 0.3, 'reg_lambda': 1.5,
        'scale_pos_weight': 1,
        'eval_metric': 'auc', 'random_state': 42,
        'tree_method': 'hist', 'device': 'cuda'
    }

    m = XGBClassifier(**params)
    m.fit(X_train_us, y_train_us)

    train_auc = roc_auc_score(y_train_us, m.predict_proba(X_train_us)[:, 1])
    val_auc   = roc_auc_score(y_val,      m.predict_proba(X_val)[:, 1])

    mlflow.log_params(params)
    mlflow.log_param("balancing_method",  "RandomUnderSampling")
    mlflow.log_param("sampling_strategy", 0.1)
    mlflow.log_metric("train_rows_after", X_train_us.shape[0])
    mlflow.log_metric("train_auc",        train_auc)
    mlflow.log_metric("val_auc",          val_auc)
    mlflow.log_metric("overfit_gap",      train_auc - val_auc)

    print(f"\nUNDERSAMPLING — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"Gap: {train_auc - val_auc:.4f}")

Before — fraud: 16,599, non-fraud: 455,833
After  — fraud: 16,599, non-fraud: 165,990
New shape: (182589, 180)

UNDERSAMPLING — train: 0.9756, val: 0.9199
Gap: 0.0557
🏃 View run XGBoost_Undersampling at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/dc31c37607704f60a6cdce0b3d256d97
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1


In [26]:
with mlflow.start_run(run_name="XGBoost_NearMiss"):

    from imblearn.under_sampling import NearMiss

    nm = NearMiss(
        sampling_strategy=0.1,
        version=1,
        n_jobs=-1
    )
    X_train_nm, y_train_nm = nm.fit_resample(X_train, y_train)

    print(f"After NearMiss — shape: {X_train_nm.shape}")

    params = {
        'max_depth': 6, 'n_estimators': 3772,
        'learning_rate': 0.01, 'min_child_weight': 20,
        'subsample': 0.8, 'colsample_bytree': 0.8,
        'reg_alpha': 0.3, 'reg_lambda': 1.5,
        'scale_pos_weight': 1,
        'eval_metric': 'auc', 'random_state': 42,
        'tree_method': 'hist', 'device': 'cuda'
    }

    m = XGBClassifier(**params)
    m.fit(X_train_nm, y_train_nm)

    train_auc = roc_auc_score(y_train_nm, m.predict_proba(X_train_nm)[:, 1])
    val_auc   = roc_auc_score(y_val,      m.predict_proba(X_val)[:, 1])

    mlflow.log_params(params)
    mlflow.log_param("balancing_method",  "NearMiss_v1")
    mlflow.log_param("sampling_strategy", 0.1)
    mlflow.log_metric("train_rows_after", X_train_nm.shape[0])
    mlflow.log_metric("train_auc",        train_auc)
    mlflow.log_metric("val_auc",          val_auc)
    mlflow.log_metric("overfit_gap",      train_auc - val_auc)

    print(f"\nNEARMISS — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"Gap: {train_auc - val_auc:.4f}")

After NearMiss — shape: (182589, 180)

NEARMISS — train: 0.9878, val: 0.8606
Gap: 0.1272
🏃 View run XGBoost_NearMiss at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/d673887cf15f4acea86c305f4354ea43
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1


In [27]:
with mlflow.start_run(run_name="XGBoost_Undersample_EarlyStopping"):

    from imblearn.under_sampling import RandomUnderSampler

    rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
    X_train_us, y_train_us = rus.fit_resample(X_train, y_train)
    print(f"After undersampling — shape: {X_train_us.shape}")

    params = {
        'max_depth': 6,
        'n_estimators': 7000, 
        'learning_rate': 0.01,
        'min_child_weight': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.3,
        'reg_lambda': 1.5,
        'scale_pos_weight': 1,
        'eval_metric': 'auc',
        'random_state': 42,
        'tree_method': 'hist',
        'device': 'cuda',
        'early_stopping_rounds': 100
    }

    m = XGBClassifier(**params)
    m.fit(
        X_train_us, y_train_us,
        eval_set=[(X_val, y_val)],
        verbose=200
    )

    best_iteration = m.best_iteration
    train_auc = roc_auc_score(y_train_us, m.predict_proba(X_train_us)[:, 1])
    val_auc   = roc_auc_score(y_val,      m.predict_proba(X_val)[:, 1])

    mlflow.log_params(params)
    mlflow.log_param("balancing_method",   "RandomUnderSampling")
    mlflow.log_param("sampling_strategy",  0.1)
    mlflow.log_metric("train_rows_after",  X_train_us.shape[0])
    mlflow.log_metric("best_iteration",    best_iteration)
    mlflow.log_metric("train_auc",         train_auc)
    mlflow.log_metric("val_auc",           val_auc)
    mlflow.log_metric("overfit_gap",       train_auc - val_auc)

    print(f"\nUNDERSAMPLE + EARLY STOP — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"Best iteration: {best_iteration}")
    print(f"Gap: {train_auc - val_auc:.4f}")

After undersampling — shape: (182589, 180)
[0]	validation_0-auc:0.80830
[200]	validation_0-auc:0.86978
[400]	validation_0-auc:0.88661
[600]	validation_0-auc:0.89463
[800]	validation_0-auc:0.89961
[1000]	validation_0-auc:0.90328
[1200]	validation_0-auc:0.90588
[1400]	validation_0-auc:0.90817
[1600]	validation_0-auc:0.90995
[1800]	validation_0-auc:0.91151
[2000]	validation_0-auc:0.91293
[2200]	validation_0-auc:0.91412
[2400]	validation_0-auc:0.91531
[2600]	validation_0-auc:0.91623
[2800]	validation_0-auc:0.91705
[3000]	validation_0-auc:0.91779
[3200]	validation_0-auc:0.91853
[3400]	validation_0-auc:0.91919
[3600]	validation_0-auc:0.91947
[3800]	validation_0-auc:0.91997
[4000]	validation_0-auc:0.92044
[4200]	validation_0-auc:0.92107
[4400]	validation_0-auc:0.92147
[4600]	validation_0-auc:0.92181
[4800]	validation_0-auc:0.92215
[5000]	validation_0-auc:0.92248
[5200]	validation_0-auc:0.92271
[5400]	validation_0-auc:0.92291
[5600]	validation_0-auc:0.92309
[5800]	validation_0-auc:0.92322
[600

In [28]:
with mlflow.start_run(run_name="XGBoost_Undersample_EarlyStopping"):

    from imblearn.under_sampling import RandomUnderSampler

    rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
    X_train_us, y_train_us = rus.fit_resample(X_train, y_train)
    print(f"After undersampling — shape: {X_train_us.shape}")

    params = {
        'max_depth': 8,
        'min_child_weight': 5, 
        'gamma': 0.2,     
        'learning_rate': 0.01,
        'n_estimators': 12000, 
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.1,
        'reg_lambda': 2.0,
        'grow_policy': 'lossguide',
        'max_leaves': 64,
        'scale_pos_weight': 1,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'auc',
        'random_state': 42,
        'early_stopping_rounds': 100

    }

    m = XGBClassifier(**params)
    m.fit(
        X_train_us, y_train_us,
        eval_set=[(X_val, y_val)],
        verbose=200
    )

    best_iteration = m.best_iteration
    train_auc = roc_auc_score(y_train_us, m.predict_proba(X_train_us)[:, 1])
    val_auc   = roc_auc_score(y_val,      m.predict_proba(X_val)[:, 1])

    mlflow.log_params(params)
    mlflow.log_param("balancing_method",   "RandomUnderSampling")
    mlflow.log_param("sampling_strategy",  0.1)
    mlflow.log_metric("train_rows_after",  X_train_us.shape[0])
    mlflow.log_metric("best_iteration",    best_iteration)
    mlflow.log_metric("train_auc",         train_auc)
    mlflow.log_metric("val_auc",           val_auc)
    mlflow.log_metric("overfit_gap",       train_auc - val_auc)

    print(f"\nUNDERSAMPLE + EARLY STOP — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"Best iteration: {best_iteration}")
    print(f"Gap: {train_auc - val_auc:.4f}")

After undersampling — shape: (182589, 180)
[0]	validation_0-auc:0.81014
[200]	validation_0-auc:0.87863
[400]	validation_0-auc:0.89890
[600]	validation_0-auc:0.90720
[800]	validation_0-auc:0.91139
[1000]	validation_0-auc:0.91464
[1200]	validation_0-auc:0.91702
[1400]	validation_0-auc:0.91934
[1600]	validation_0-auc:0.92070
[1800]	validation_0-auc:0.92184
[2000]	validation_0-auc:0.92313
[2200]	validation_0-auc:0.92401
[2400]	validation_0-auc:0.92459
[2600]	validation_0-auc:0.92524
[2800]	validation_0-auc:0.92575
[3000]	validation_0-auc:0.92617
[3200]	validation_0-auc:0.92643
[3400]	validation_0-auc:0.92661
[3600]	validation_0-auc:0.92683
[3800]	validation_0-auc:0.92707
[3957]	validation_0-auc:0.92708

UNDERSAMPLE + EARLY STOP — train: 0.9932, val: 0.9271
Best iteration: 3857
Gap: 0.0661
🏃 View run XGBoost_Undersample_EarlyStopping at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/44f68d18ce6745db89d8b5de854fc1a5
🧪 View experiment at: https://dagshub.com/gvakh23/M

In [29]:
with mlflow.start_run(run_name="XGBoost_EarlyStopping_complex"):

    params = {
        'max_depth': 8,
        'min_child_weight': 5,
        'gamma': 0.2,   
        'learning_rate': 0.01,
        'n_estimators': 12000, 
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.1,
        'reg_lambda': 2.0,
        'grow_policy': 'lossguide',
        'max_leaves': 64,
        'scale_pos_weight': scale_pos_weight,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'auc',
        'random_state': 42,
        'early_stopping_rounds': 100

    }

    m = XGBClassifier(**params)
    m.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=200
    )

    best_iteration = m.best_iteration
    train_auc = roc_auc_score(y_train, m.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,      m.predict_proba(X_val)[:, 1])

    mlflow.log_params(params)
    mlflow.log_param("balancing_method",   "scale_pos_weight")
    mlflow.log_param("feature_engineering",   None)

    mlflow.log_param("sampling_strategy",  0.1)
    mlflow.log_metric("train_rows_after",  X_train.shape[0])
    mlflow.log_metric("best_iteration",    best_iteration)
    mlflow.log_metric("train_auc",         train_auc)
    mlflow.log_metric("val_auc",           val_auc)
    mlflow.log_metric("overfit_gap",       train_auc - val_auc)

    print(f"\nUNDERSAMPLE + EARLY STOP — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"Best iteration: {best_iteration}")
    print(f"Gap: {train_auc - val_auc:.4f}")

[0]	validation_0-auc:0.82853
[200]	validation_0-auc:0.88628
[400]	validation_0-auc:0.90292
[600]	validation_0-auc:0.91042
[800]	validation_0-auc:0.91458
[1000]	validation_0-auc:0.91697
[1200]	validation_0-auc:0.91847
[1400]	validation_0-auc:0.91988
[1600]	validation_0-auc:0.92057
[1800]	validation_0-auc:0.92153
[2000]	validation_0-auc:0.92198
[2174]	validation_0-auc:0.92204

UNDERSAMPLE + EARLY STOP — train: 0.9884, val: 0.9223
Best iteration: 2074
Gap: 0.0661
🏃 View run XGBoost_EarlyStopping_complex at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/74fef092ff804875ae2760d30ada5d93
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1


In [30]:
with mlflow.start_run(run_name="XGBoost_Undersample_EarlyStopping"):

    from imblearn.under_sampling import RandomUnderSampler

    rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
    X_train_us, y_train_us = rus.fit_resample(X_train, y_train)
    print(f"After undersampling — shape: {X_train_us.shape}")

    params = {
        'max_depth': 8,
        'min_child_weight': 5,
        'gamma': 0.2,
        'learning_rate': 0.01,
        'n_estimators': 12000,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.1,
        'reg_lambda': 2.0,
        'grow_policy': 'lossguide',
        'max_leaves': 64,
        'scale_pos_weight': 1,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'auc',
        'random_state': 42,
        'early_stopping_rounds': 100
    }

    m = XGBClassifier(**params)
    m.fit(
        X_train_us, y_train_us,
        eval_set=[(X_val, y_val)],
        verbose=200
    )

    best_iteration = m.best_iteration
    train_auc = roc_auc_score(y_train_us, m.predict_proba(X_train_us)[:, 1])
    val_auc   = roc_auc_score(y_val,      m.predict_proba(X_val)[:, 1])

    mlflow.log_params(params)
    mlflow.log_param("balancing_method",   "RandomUnderSampling")
    mlflow.log_param("sampling_strategy",  0.1)
    mlflow.log_metric("train_rows_after",  X_train_us.shape[0])
    mlflow.log_metric("best_iteration",    best_iteration)
    mlflow.log_metric("train_auc",         train_auc)
    mlflow.log_metric("val_auc",           val_auc)
    mlflow.log_metric("overfit_gap",       train_auc - val_auc)

    print(f"\nUNDERSAMPLE + EARLY STOP — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"Best iteration: {best_iteration}")
    print(f"Gap: {train_auc - val_auc:.4f}")

After undersampling — shape: (182589, 180)
[0]	validation_0-auc:0.81014
[200]	validation_0-auc:0.87863
[400]	validation_0-auc:0.89890
[600]	validation_0-auc:0.90720
[800]	validation_0-auc:0.91139
[1000]	validation_0-auc:0.91464
[1200]	validation_0-auc:0.91702
[1400]	validation_0-auc:0.91934
[1600]	validation_0-auc:0.92070
[1800]	validation_0-auc:0.92184
[2000]	validation_0-auc:0.92313
[2200]	validation_0-auc:0.92401
[2400]	validation_0-auc:0.92459
[2600]	validation_0-auc:0.92524
[2800]	validation_0-auc:0.92575
[3000]	validation_0-auc:0.92617
[3200]	validation_0-auc:0.92643
[3400]	validation_0-auc:0.92661
[3600]	validation_0-auc:0.92683
[3800]	validation_0-auc:0.92707
[3957]	validation_0-auc:0.92708

UNDERSAMPLE + EARLY STOP — train: 0.9932, val: 0.9271
Best iteration: 3857
Gap: 0.0661
🏃 View run XGBoost_Undersample_EarlyStopping at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/3f2c9a57730a45caa573a15372583a93
🧪 View experiment at: https://dagshub.com/gvakh23/M

In [41]:
from sklearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

class TargetDropper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.feature_columns_ = [c for c in X.columns
                                  if c not in ['isFraud', 'TransactionID']]
        return self
    def transform(self, X):
        df = X.drop(columns=['isFraud', 'TransactionID'], errors='ignore').copy()
        if hasattr(self, 'feature_columns_'):
            for col in self.feature_columns_:
                if col not in df.columns:
                    df[col] = -999
            df = df[self.feature_columns_]
        return df

with mlflow.start_run(run_name='XGBoost_FullPipeline_Registry') as run:
    print('Step 1: Fitting preprocessors on full 590k rows...')

    cleaner_f  = Cleaner(null_threshold=0.9, fill_value=-999)
    encoder_f  = CategoricalEncoder()
    engineer_f = FeatureEngineer()
    dropper_f  = TargetDropper()
    selector_f = FeatureSelector(variance_threshold=0.01)

    _c    = cleaner_f.fit_transform(train, train['isFraud'])
    _e    = encoder_f.fit_transform(_c)
    _fe   = engineer_f.fit_transform(_e, train['isFraud'])
    dropper_f.fit(_fe) 
    _fe_x = dropper_f.transform(_fe)
    y_full = train['isFraud'].reset_index(drop=True)
    X_full = selector_f.fit_transform(_fe_x, y_full).reset_index(drop=True)
    print(f'Processed shape: {X_full.shape}')

    # ── Undersample ───────────────────────────────────────
    print('Step 2: Undersampling...')
    rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
    X_us, y_us = rus.fit_resample(X_full, y_full)
    print(f'After undersampling: {X_us.shape}')

    # ── Train final model with best params ────────────────
    # n_estimators=3857 = optimal from early stopping experiment
    print('Step 3: Training final model...')
    best_params = {
        'max_depth':        8,
        'min_child_weight': 5,
        'gamma':            0.2,
        'learning_rate':    0.01,
        'n_estimators':     3857,
        'subsample':        0.85,
        'colsample_bytree': 0.85,
        'reg_alpha':        0.1,
        'reg_lambda':       2.0,
        'grow_policy':      'lossguide',
        'max_leaves':       64,
        'scale_pos_weight': 1,
        'tree_method':      'hist',
        'device':           'cuda',
        'eval_metric':      'auc',
        'random_state':     42,
    }
    final_xgb = XGBClassifier(**best_params)
    final_xgb.fit(X_us, y_us)
    train_auc = roc_auc_score(y_us, final_xgb.predict_proba(X_us)[:, 1])
    print(f'Train AUC on undersampled: {train_auc:.4f}')

    # ── Build full pipeline ───────────────────────────────
    print('Step 4: Building pipeline...')
    full_pipeline = Pipeline([
        ('cleaner',  cleaner_f),
        ('encoder',  encoder_f),
        ('engineer', engineer_f),
        ('dropper',  dropper_f),
        ('selector', selector_f),
        ('model',    final_xgb),
    ])

    # ── Verify pipeline works on raw data ─────────────────
    print('Step 5: Verifying pipeline on raw sample...')
    raw_sample = train.iloc[:100].drop(columns=['isFraud'])
    test_preds = full_pipeline.predict_proba(raw_sample)[:, 1]
    print(f'Sample preds: min={test_preds.min():.3f}, max={test_preds.max():.3f}')
    print('Pipeline verified on raw data!')

    # ── Log and save ──────────────────────────────────────
    mlflow.log_params(best_params)
    mlflow.log_param('pipeline_steps',
                     'Cleaner→CategoricalEncoder→FeatureEngineer→TargetDropper→FeatureSelector→XGB')
    mlflow.log_param('trained_on',       'full_590k_rows')
    mlflow.log_param('balancing',        'RandomUnderSampling_0.1')
    mlflow.log_param('n_estimators_src', 'early_stopping_best_iter_3857')
    mlflow.log_param('can_run_on_raw',   True)
    mlflow.log_metric('train_auc',       train_auc)
    mlflow.log_metric('val_auc',         0.9271)
    mlflow.log_metric('overfit_gap',     0.0661)
    mlflow.log_metric('n_features',      X_full.shape[1])

    mlflow.sklearn.log_model(
        sk_model=full_pipeline,
        name="full_xgboost_pipeline"
    )
    xgb_run_id = run.info.run_id
    print(f'\nFull pipeline saved!')
    print(f'Run ID: {xgb_run_id}')



Step 1: Fitting preprocessors on full 590k rows...
clean_in
🏃 View run XGBoost_Cleaning at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/9b426373bd9349f3b14f4fa90851df6c
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1
[Cleaner] Done — dropped 12 cols, 590540 rows
clean trans in
encode_in
🏃 View run XGBoost_Categorical_Encoding at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/e5a6de2781134e0c871c62c1e07f7206
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1
[CategoricalEncoder] Done — OHE cols: 13, LE cols: 6
encode trans in
select_in
🏃 View run XGBoost_Feature_Selection at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/0ca0acf62dde498297aa6c376f88ed7b
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1
[FeatureSelector] Done — dropped 325 features, 181 remaining
select trans in
Processed shape: (59

2026/05/05 20:19:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Full pipeline saved!
Run ID: 919c07cb683540d089ad994e9eab51f9
🏃 View run XGBoost_FullPipeline_Registry at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1/runs/919c07cb683540d089ad994e9eab51f9
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/1
